In [31]:
from rdflib import Graph, Namespace, RDF, RDFS, OWL, Literal, URIRef

In [ ]:
# load rdf/sample.ttl
g = Graph()
g.parse("../rdf/samples.ttl", format="ttl")
g.parse("../rdf/observations.ttl", format="ttl")
print(f"Graph has {len(g)} statements.")

reused_ontologies = Graph()
reused_ontologies.parse("https://www.w3.org/ns/prov-o", format="ttl")
reused_ontologies.parse(
    "https://github.com/w3c/sdw-sosa-ssn/raw/refs/heads/gh-pages/ssn/rdf/ontology/core/sosa-common.ttl",
    format="ttl",
)
# reused_ontologies.parse(
#    "https://data.bioontology.org/ontologies/SSN/submissions/2/download?apikey=8b5b7825-538d-40e0-9e9e-5ab9274a9aeb",
#    format="owl",
# )

Graph has 2889 statements.


<Graph identifier=N48788deca4be43b1aaaa5ad8ca1e9395 (<class 'rdflib.graph.Graph'>)>

In [46]:
# add prefix namespaces
# @prefix sosa: <http://www.w3.org/ns/sosa/> .
# @prefix schema: <https://schema.org/> .
# @prefix abromics: <https://abromics.fr/> .
# @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
# @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
# @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
# @prefix obo: <http://purl.obolibrary.org/obo/> .
# @prefix aro: <http://purl.obolibrary.org/obo/ARO_> .
# @prefix go: <http://purl.org/obo/owl/GO#> .
# @prefix uniprot: <http://purl.uniprot.org/uniprot/> .
# @prefix uniprot-core: <http://purl.uniprot.org/uniprot/core/> .
# @prefix geo: <http://www.w3.org/2003/01/geo/wgs84_pos#> .
# @prefix sio: <http://semanticscience.org/resource/> .
# @prefix skos: <http://www.w3.org/2004/02/skos/core#> .
# @prefix prov: <http://www.w3.org/ns/prov#> .
# @prefix foaf: <http://xmlns.com/foaf/0.1/> .
# @prefix wdt: <http://www.wikidata.org/prop/direct/> .

sosa = Namespace("http://www.w3.org/ns/sosa/")
g.bind("sosa", sosa)
schema = Namespace("https://schema.org/")
g.bind("schema", schema)
abromics = Namespace("https://abromics.fr/")
g.bind("abromics", abromics)
rdf = Namespace("http://www.w3.org/1999/02/22-rdf-syntax-ns#")
g.bind("rdf", rdf)
rdfs = Namespace("http://www.w3.org/2000/01/rdf-schema#")
g.bind("rdfs", rdfs)
xsd = Namespace("http://www.w3.org/2001/XMLSchema#")
g.bind("xsd", xsd)
obo = Namespace("http://purl.obolibrary.org/obo/")
g.bind("obo", obo)
aro = Namespace("http://purl.obolibrary.org/obo/ARO_")
g.bind("aro", aro)
go = Namespace("http://purl.org/obo/owl/GO#")
g.bind("go", go)
uniprot = Namespace("http://purl.uniprot.org/uniprot/")
g.bind("uniprot", uniprot)
uniprot_core = Namespace("http://purl.uniprot.org/uniprot/core/")
g.bind("uniprot-core", uniprot_core)
geo = Namespace("http://www.w3.org/2003/01/geo/wgs84_pos#")
g.bind("geo", geo)
sio = Namespace("http://semanticscience.org/resource/")
g.bind("sio", sio)
skos = Namespace("http://www.w3.org/2004/02/skos/core#")
g.bind("skos", skos)
prov = Namespace("http://www.w3.org/ns/prov#")
g.bind("prov", prov)
foaf = Namespace("http://xmlns.com/foaf/0.1/")
g.bind("foaf", foaf)
wdt = Namespace("http://www.wikidata.org/prop/direct/")
g.bind("wdt", wdt)
abromics_kg = Namespace("https://abromics.fr/abromics-kg/")
g.bind("abromics-kg", abromics_kg)

# iterate over all rdf:type properties to find all classes
classes = set()
for s, p, o in g.triples((None, RDF.type, None)):
    classes.add(o)
print("Classes in the graph:")
print(classes)

# iterate over all properties to find their domains and ranges
properties = set()
for s, p, o in g:
    properties.add(p)

Classes in the graph:
{rdflib.term.URIRef('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C12390'), rdflib.term.URIRef('http://semanticscience.org/resource/000000'), rdflib.term.URIRef('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C13283'), rdflib.term.URIRef('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C12428'), rdflib.term.URIRef('http://semanticscience.org/resource/001050'), rdflib.term.URIRef('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C12736'), rdflib.term.URIRef('http://www.w3.org/ns/prov#Entity'), rdflib.term.URIRef('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C13234'), rdflib.term.URIRef('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C12468'), rdflib.term.URIRef('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C13278')}


In [47]:
rdfs_schema = Graph()
for cls in classes:
    # create an instance of the class using owl:Class
    rdfs_schema.add((cls, RDF.type, OWL.Class))

# RDF triples to create an insatnce of an owl:Ontology
ontology_uri = URIRef("https://abromics.fr/ontology/abromics-kg-inferred-schema")
rdfs_schema.add((ontology_uri, RDF.type, OWL.Ontology))
rdfs_schema.add((
    ontology_uri,
    RDFS.label,
    Literal("ABRomics Knowledge Graph Inferred Schema"),
))
rdfs_schema.add((
    ontology_uri,
    RDFS.comment,
    Literal(
        "This ontology represents the inferred schema from the ABRomics Knowledge Graph."
    ),
))
# import datetime to add the date of creation
from datetime import datetime

rdfs_schema.add((
    ontology_uri,
    OWL.versionInfo,
    Literal(f"Created on {datetime.now().strftime('%Y-%m-%d')}"),
))


# serialize the schema graph to turtle format
schema_ttl = rdfs_schema.serialize(format="turtle")
print("Inferred schema in Turtle format:")
print(schema_ttl)


Inferred schema in Turtle format:
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

<http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C12390> a owl:Class .

<http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C12428> a owl:Class .

<http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C12468> a owl:Class .

<http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C12736> a owl:Class .

<http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C13234> a owl:Class .

<http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C13278> a owl:Class .

<http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C13283> a owl:Class .

<http://semanticscience.org/resource/000000> a owl:Class .

<http://semanticscience.org/resource/001050> a owl:Class .

prov:Entity a owl:Class .

<https://abromics.fr/ontology/abromics-kg-inferred-schema> a owl:Ontology ;
    rdfs:label "ABRomics Knowledge Graph Inferred Schema" ;
    rdfs

In [48]:
for p in properties:
    # describe query to get domain and range
    query = f"""
    DESCRIBE <{p}>
    """
    print(f"Querying for property: {p}")
    result_graph = reused_ontologies.query(query).graph
    if len(result_graph) > 0:
        print(f"Property: {p}")
        print(result_graph.serialize(format="turtle"))
        rdfs_schema += result_graph


Querying for property: http://www.w3.org/1999/02/22-rdf-syntax-ns#type
Querying for property: http://www.w3.org/ns/prov#atLocation
Property: http://www.w3.org/ns/prov#atLocation
@prefix : <http://www.w3.org/ns/prov#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:atLocation a owl:ObjectProperty ;
    rdfs:label "atLocation" ;
    rdfs:comment "This property has multiple RDFS domains to suit multiple OWL Profiles. See <a href=\"#owl-profile\">PROV-O OWL Profile</a>.",
        "The Location of any resource."@en ;
    rdfs:domain [ a owl:Class ;
            owl:unionOf ( :Activity :Agent :Entity :InstantaneousEvent ) ] ;
    rdfs:isDefinedBy <http://www.w3.org/ns/prov-o#> ;
    rdfs:range :Location ;
    :category "expanded" ;
    :editorialNote "The naming of prov:atLocation parallels prov:atTime, and is not named prov:hadLocation to avoid conflicting with the convent

In [49]:
rdfs_schema.serialize(destination="inferred_schema.ttl", format="turtle")

<Graph identifier=Ne304e11ab304490d8ac409a435d6e91c (<class 'rdflib.graph.Graph'>)>

In [50]:
#!pip install pylode

In [51]:
# launch pylode to visualize the schema
!pylode inferred_schema.ttl -o inferred_schema.html 

INFO:root:Loading background ontologies from a pickle file
DEBUG:asyncio:Using selector: KqueueSelector
